# Gym Interface Tutorial

The CloudPendulum gym interface lets you reserve and control **multiple hardware cells in parallel** under a single session. This is particularly useful for:

- **Reinforcement learning**: run many independent environments simultaneously; start a new episode on the same reserved cells without re-joining the queue between episodes
- **System identification**: excite several pendulums simultaneously from different initial conditions and fit a model to the ensemble of trajectories
- **Physical domain randomization**: exploit real hardware-to-hardware differences to build controllers that generalize across physical systems

The connection can happen through two different mechanisms:
1. The gym interface that is discussed here. This gives you access to all low-level controls that the CloudPendulum exposes and is suited for general multi-platform usage.
2. A class mimicking a [Gymnasium environment](https://gymnasium.farama.org/index.html) that is suitable for Reinforcement Learning tasks specifically. For the latter, please refer to [this Jupyter Notebook](https://github.com/air-course/tutorial14/blob/main/Part_I/tutorial14_teachers.ipynb) as an introduction and [these files](https://github.com/air-course/tutorial14/tree/main/Part_I/pendulum_plant) to run the gymnasium environment.

In this tutorial we identify a simple pendulum model from free-fall data. We reserve three cells at once, start each from a different initial angle, let them swing freely, and fit the equation of motion to the collected trajectories. Running a second batch of episodes (without re-joining the queue) shows how the parameter estimates converge with more data.

## Setup


In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import os
from typing import List, Tuple

from cloudpendulumclient.client import Client
from cloudpendulumclient.data import CellType

Your user token should be in `~/.env` (one line, no quotes):

```bash
echo "your_token_here" > ~/.env
```

You can also pass the token directly as a string in your code, but please make sure not to commit it to a public repository.

**Important:** the gym interface requires a token with gym permissions. Standard experiment tokens do not grant access to `start_gym()`. You can check wether you have gym token at

https://cloudpendulum.m2.chalmers.se:1443/hub/hardware-status

and request a gym-enabled token at

https://cloudpendulum.m2.chalmers.se:1443/hub/token-request .

In [ ]:
with open(os.path.expanduser("~/.env"), 'r') as f:
    user_token = f.read().strip()

client = Client()

info = client.get_user_info(user_token)
print(f"Attempts left:    {info.number_of_attempts}")
print(f"Max gym length:   {info.max_gym_length} s")
print(f"Gym bucket size:  {info.gym_bucket_size} / {info.gym_max_bucket_size}")

In [ ]:
# Gym & episode settings 
N_CELLS          = 2    # number of cells to reserve simultaneously
GYM_TIME         = 30   # total gym reservation [s]
EPISODE_TIME     = 5    # duration of each episode [s]
PREPARATION_TIME = 4    # countdown before each episode starts [s]

# Control loop 
DT = 0.02   # control period [s]  ->  50 Hz

# Physical constants 
GRAVITY = 9.81   # m/s^2

n_steps = int(EPISODE_TIME / DT)

## Gym Session and Episodes

`start_gym()` reserves a block of hardware cells under a single lease. All subsequent `start_episode()` calls draw from that pool and work exactly like `start_experiment()` - you can set impedance parameters, read position and velocity, send torques, configure safety limits, etc. The only difference is the token hierarchy:

1. **`user_token`** comes from `~/.env` and is used for `start_gym()`
2. **`gym_token`** comes from `start_gym()` return value and is used for `start_episode()`, `stop_gym()`
3. **`episode_token`** comes from `start_episode()` first return value and is used for all per-cell calls: `get_position()`, `set_torque()`, `stop_episode()`, ... within a single episode.

This means that for a gym with four episodes at the same time, you have one gym token and four episode tokes that you need to keep track of. 

## Free-fall Data Collection and Parameter Estimation

Before we start the gym and do the experiments, let us quickly discuss what is happening in the following: We do a simple system identification of the inverted pendulum. The details are explained in the following, but you can skip this if you are only interested into the gym syntax.

Setting `Kp = Kd = 0` puts the motor in torque mode with zero torque — the pendulum swings freely under gravity and friction. We send no torque commands; the `get_position()` and `get_velocity()` calls inside the loop serve as heartbeat requests that keep the session alive.

The equation of motion of a simple pendulum is:

```
I * q_ddot + b * q_dot + m*g*l * sin(q) = 0
```

Rearranging with `p1 = b/I` and `p2 = m*g*l/I = g/l` (using `I = m*l^2` for a point-mass pendulum):

```
q_ddot = -p1 * q_dot - p2 * sin(q)
```

This is linear in `[p1, p2]`, so we can fit both parameters simultaneously via least squares given measured positions, velocities, and numerically differentiated accelerations. The pendulum length then follows directly:

```
l = g / p2
```

The `fit_parameters` helper below will be reused for doing the actual system identification.

In [ ]:
def fit_parameters(pos, vel, dt):
    """Fit p1 = b/I and p2 = g/l via least squares. Returns (p1, p2, l_est)."""
    # Central-difference acceleration, skipping the first and last sample
    acc = (vel[:, 2:] - vel[:, :-2]) / (2 * dt)
    q   = pos[:, 1:-1].ravel()
    qd  = vel[:, 1:-1].ravel()
    qdd = acc.ravel()

    # Regressor:  q_ddot = -p1 * q_dot - p2 * sin(q)
    Phi = np.column_stack([-qd, -np.sin(q)])
    params, _, _, _ = np.linalg.lstsq(Phi, qdd, rcond=None)
    p1, p2 = params

    return float(p1), float(p2), float(GRAVITY / p2)

## Running Gym Experiments

We start by reserving a cell using `start_gym` which returns the gym token. It also prints a livestream URL that shows all reserved cells side by side, which can be useful to monitor the whole gym at a glance.

While the gym session is running, we can start multiple episodes. Here, we start each episode in a separate thread so that they are independent of each other. For this, we create a new client instance and pass the gym token.

Inside the thread, starting episodes works exactly like with `start_experiment()`, except you use the gym token. Every episode gets a unique **episode token**, which is then used for all subsequent interactions with that cell. This can be e.g. `get_position()`, `get_velocity()` or at the end `stop_experiment()` (please refer to the [documentation](https://cloudpendulum.m2.chalmers.se/docs/) for a full overview).

As long as the gym session is active, you can start more episodes immediately. This is one of the key advantages of the gym interface: the hardware stays reserved between episodes, so a new run begins as soon as the previous one ends.

At the end, `stop_gym()` releases all reserved cells and closes the session.

We first start by defining the callback function that starts an individual episode:

In [ ]:
import threading

def run_episode(
    gym_token: str,
    angle: float,
    results: List[Tuple[np.ndarray, np.ndarray]],
    lock: threading.Lock,
) -> None:
    """
    Runs one complete episode:
      1. start episode
      2. prepare (set controller params)
      3. collect pos/vel at fixed rate
      4. stop episode
      5. append (pos, vel) to shared results list under the lock
    """
    thread_client = Client()
    pos = np.zeros(n_steps)
    vel = np.zeros(n_steps)

    ept = time.time()
    # 1. Start episode
    token, url = thread_client.start_episode(
        gym_token, "SimplePendulum", EPISODE_TIME, 0.0,
        initial_position=[angle],
    )
    print(f"[{threading.current_thread().name}]  q0={np.degrees(angle):.0f}°  token={token}")

    print(f"eps start took {time.time() - ept}")

    try:
        # 2. Prepare
        thread_client.set_impedance_controller_params([0.0], [0.0], token)

        # 3. Collect at fixed rate
        t_start = time.time()
        for i in range(n_steps):
            pos[i] = thread_client.get_position(token)
            vel[i] = thread_client.get_velocity(token)
            sleep_time = t_start + (i + 1) * DT - time.time()
            if sleep_time > 0:
                time.sleep(sleep_time)
        
        print(f"episode took {time.time() - t_start} while having {EPISODE_TIME}")

        # 4. Stop
        thread_client.stop_episode(token)

    except Exception as exc:
        print(f"[{threading.current_thread().name}]  error: {exc}")
        try:
            thread_client.stop_episode(token)
        except Exception:
            pass

    # 5. Append result (thread-safe)
    with lock:
        results.append((pos.copy(), vel.copy()))

and then run the gym, internally calling `run_episode` multiple times:

In [ ]:
gym_token = client.start_gym(user_token, GYM_TIME, [CellType.SINGLE] * N_CELLS)

results: List[Tuple[np.ndarray, np.ndarray]] = []
lock      = threading.Lock()
semaphore = threading.Semaphore(N_CELLS)   # cap live threads
threads   = []

number_of_batches = int(np.floor(GYM_TIME / (EPISODE_TIME + PREPARATION_TIME)))
number_of_experiments = number_of_batches * N_CELLS
angles = np.random.uniform(low=-np.pi, high=np.pi, size=(number_of_experiments,))

system_params = []

experiments_started = 0

while experiments_started < len(angles):
    angle = angles[experiments_started]
    semaphore.acquire() # blocks if N_CONCURRENT threads are already live

    def _target(a=angle):
        try:
            run_episode(gym_token, a, results, lock)
        finally:
            semaphore.release()  # free the slot when the thread finishes

    t = threading.Thread(target=_target, name=f"exp-{experiments_started:03d}")
    t.start()
    threads.append(t)
    experiments_started += 1

# Wait for all remaining threads to finish
for t in threads:
    t.join()


After having collected all data, we can run the identification:

In [ ]:
pos_all = np.array([r[0] for r in results])
vel_all = np.array([r[1] for r in results])

p1, p2, l = fit_parameters(pos_all, vel_all, DT)

print(f"p1: {p1}")
print(f"p2: {p2}")
print(f"Pendulum length: {l}")

The actual length of the pendulum is 10cm, so the results are pretty accurate. While it presumably would also have worked with less data, the gym environment allowed us to collect four times the data at the same time compared to the `start_experiment` syntax.

## Episodes Don't Have to Be Synchronised

The experiments above started all episodes at the same time. But `start_episode()` can be called at any time while the gym session is active, i.e. each episode is fully independent of any others. This makes it straightforward to drive one episode at a time, for example in a reinforcement learning training loop:

```python
for episode in range(N_EPISODES):
    token, url = client.start_episode(gym_token, "SimplePendulum", EPISODE_TIME)

    reward = run_policy(client, token)   # run your policy, collect reward

    client.stop_episode(token)
    update_policy(reward)                # process before starting the next episode
```

> For Reinforcement Learning specifically, we also have the **Gym Interface** that exposes the hardware as a [Gymnasium](https://gymnasium.farama.org/index.html) class. This integrates even tighter into existing RL pipelines.

## What Next?

Feel free to design your own experiments to your likings. In case you are interested into the gymnasium wrapper class, you can take a look [here](https://github.com/air-course/tutorial14/blob/main/Part_I/tutorial14_teachers.ipynb).